# 🎓 Smart Classroom — YOLOv8 Object Detector
### student + janitor detection | ~10-15 Minutes Training on T4 GPU
**Workflow:**
1. Mount Google Drive (datasets already there)
2. Convert COCO JSON → YOLO TXT format automatically
3. Train YOLOv8n (nano) for 50 epochs → ~10-15 min
4. Download `best.pt` from Drive → put in `edge_app/model/`

In [ ]:
# Step 1: Install YOLOv8 & Mount Google Drive
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil, random, glob
from pathlib import Path
from collections import defaultdict

print("✅ Ultralytics YOLO installed and Drive mounted!")

In [ ]:
# Step 2: Configuration
PERSON_DIR = '/content/drive/MyDrive/classroom_dataset/person'
BUCKET_DIR = '/content/drive/MyDrive/classroom_dataset/bucket'

OUTPUT_DIR = '/content/drive/MyDrive/classroom_model_yolo'
WORK_DIR   = '/content/yolo_dataset'   # local working directory in Colab

os.makedirs(OUTPUT_DIR, exist_ok=True)

# YOLO class mapping
# 0 = student (person images)
# 1 = janitor (bucket images)
CLASS_NAMES = ['student', 'janitor']
CLASS_MAP   = {'person': 0, 'bucket': 1}

print(f"Classes: {CLASS_NAMES}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Step 3: Auto-Convert COCO JSON → YOLO TXT format
# (Roboflow exported COCO JSON, YOLO needs TXT labels)

def coco_json_to_yolo(coco_json_path, img_dir, yolo_img_dir, yolo_lbl_dir, class_id):
    """Convert _annotations.coco.json to YOLO .txt format."""
    os.makedirs(yolo_img_dir, exist_ok=True)
    os.makedirs(yolo_lbl_dir, exist_ok=True)

    with open(coco_json_path) as f:
        coco = json.load(f)

    id2info  = {img['id']: img for img in coco['images']}
    img2anns = defaultdict(list)
    for ann in coco['annotations']:
        img2anns[ann['image_id']].append(ann)

    converted = 0
    for img_id, anns in img2anns.items():
        info     = id2info[img_id]
        fname    = info['file_name']
        iw, ih   = info['width'], info['height']
        src_img  = os.path.join(img_dir, fname)

        if not os.path.exists(src_img):
            continue

        # Copy image
        dst_img = os.path.join(yolo_img_dir, fname)
        shutil.copy2(src_img, dst_img)

        # Write YOLO label
        label_fname = os.path.splitext(fname)[0] + '.txt'
        dst_lbl = os.path.join(yolo_lbl_dir, label_fname)
        lines   = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w < 2 or h < 2:
                continue
            cx = (x + w/2) / iw
            cy = (y + h/2) / ih
            nw = w / iw
            nh = h / ih
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if lines:
            with open(dst_lbl, 'w') as f:
                f.write("\n".join(lines))
            converted += 1

    return converted

# Convert all splits for both datasets
for split in ['train', 'valid', 'test']:
    img_dir_out = os.path.join(WORK_DIR, split, 'images')
    lbl_dir_out = os.path.join(WORK_DIR, split, 'labels')

    for ds_dir, ds_name in [(PERSON_DIR,'person'), (BUCKET_DIR,'bucket')]:
        ann_path  = os.path.join(ds_dir, split, '_annotations.coco.json')
        img_dir   = os.path.join(ds_dir, split)
        cls_id    = CLASS_MAP[ds_name]

        if os.path.exists(ann_path):
            n = coco_json_to_yolo(ann_path, img_dir, img_dir_out, lbl_dir_out, cls_id)
            print(f"  Converted [{split.upper()}] {ds_name}: {n} images → YOLO format")
        else:
            print(f"  SKIP: {ann_path} not found")

print("\n✅ COCO JSON → YOLO TXT conversion complete!")

In [ ]:
# Step 4: Write data.yaml for YOLO training
import yaml

data_yaml = {
    'path':  WORK_DIR,
    'train': os.path.join(WORK_DIR, 'train', 'images'),
    'val':   os.path.join(WORK_DIR, 'valid', 'images'),
    'test':  os.path.join(WORK_DIR, 'test',  'images'),
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES
}

yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml written:")
print(open(yaml_path).read())

# Count images
for split in ['train', 'valid']:
    n = len(glob.glob(os.path.join(WORK_DIR, split, 'images', '*.*')))
    print(f"  {split}: {n} images")

In [ ]:
# Step 5: Train YOLOv8 (50 epochs, ~10-15 min on T4 GPU)
from ultralytics import YOLO

# YOLOv8n = nano (fastest, still very accurate for simple detection)
model = YOLO('yolov8n.pt')

results = model.train(
    data     = yaml_path,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,          # YOLO can handle larger batches efficiently
    patience = 20,          # Early stopping if no improvement
    device   = 0,           # GPU
    project  = '/content/runs/detect',
    name     = 'classroom_v5',
    exist_ok = True,
    verbose  = True,
    plots    = True,
)

print("\n✅ Training Complete!")

In [ ]:
# Step 6: Validate & Show Metrics
from ultralytics import YOLO

best_pt = '/content/runs/detect/classroom_v5/weights/best.pt'
model   = YOLO(best_pt)

metrics = model.val(data=yaml_path, conf=0.45)
print(f"\nmAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Step 7: Export best.pt to Google Drive for download
import shutil

best_src = '/content/runs/detect/classroom_v5/weights/best.pt'
best_dst = os.path.join(OUTPUT_DIR, 'best_classroom_yolo.pt')
shutil.copy2(best_src, best_dst)

# Save class config for edge app
cfg = {
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "confidence_threshold": 0.45,
    "model_architecture": "YOLOv8n",
    "version": 5
}
cfg_dst = os.path.join(OUTPUT_DIR, 'detector_config.json')
with open(cfg_dst, 'w') as f:
    json.dump(cfg, f, indent=2)

print("=" * 50)
print("📥 DOWNLOAD THESE FILES FROM GOOGLE DRIVE:")
print(f"  Folder: {OUTPUT_DIR}")
print(f"  1. best_classroom_yolo.pt")
print(f"  2. detector_config.json")
print()
print("Copy them to your PC:")
print("  edge_app/model/best_classroom_yolo.pt")
print("  edge_app/model/detector_config.json")
print("=" * 50)